# 02 - Simulate the early-assessment cutoff
Confirm that truncating at `CUTOFF_DAY` leaves us with a meaningful
but not overwhelming amount of signal.

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Open from notebooks/ or project root.")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loader import load_all
from src.cutoff import (
    CUTOFF_DAY,
    assessments_due_by,
    submissions_by_cutoff,
    truncate_student_vle,
)

In [2]:
tables = load_all()

sv_trunc = truncate_student_vle(tables["studentVle"])
sa_trunc = submissions_by_cutoff(tables["studentAssessment"])
due = assessments_due_by(tables["assessments"])

In [3]:
print(f"CUTOFF_DAY = {CUTOFF_DAY}")
print(f"studentVle rows kept: {len(sv_trunc):,} / {len(tables['studentVle']):,} "
      f"({len(sv_trunc)/len(tables['studentVle']):.1%})")
print(f"Assessments due by cutoff: {len(due)} / {len(tables['assessments'])}")
print(f"Submissions by cutoff: {len(sa_trunc):,} / {len(tables['studentAssessment']):,} "
      f"({len(sa_trunc)/len(tables['studentAssessment']):.1%})")

CUTOFF_DAY = 28
studentVle rows kept: 2,847,851 / 10,655,280 (26.7%)
Assessments due by cutoff: 17 / 206
Submissions by cutoff: 25,614 / 173,912 (14.7%)


In [4]:
# How many assessments are typically due by the cutoff, per module–presentation?
due_per_mp = (
    due.groupby(["code_module", "code_presentation"])
    .size()
    .rename("n_assessments_due_by_cutoff")
    .reset_index()
)
display(due_per_mp.describe().round(2))

,n_assessments_due_by_cutoff
count,16.00
mean,1.06
std,0.25
min,1.00
25%,1.00
50%,1.00
75%,1.00
max,2.00
